# 07. The drone, modular: independent battery and actuator

The same MCDP as notebooks **01** and **06**, but with battery and actuator defined **independently** as their own design problems, then assembled with the `System` builder.

This is the natural way to build larger designs: each subsystem is its own DP with its own F and R, and the `System` wires them together with named algebraic constraints. The Kleene iteration converges over the joint state of all subsystem R ports.


## Imports

In [1]:
from codesign import (
    AlgebraicDP, NamedProduct, Reals, System, solve,
)

## Subsystems

Each is defined in isolation. The battery has no idea the actuator exists, and vice versa.


In [2]:
battery = AlgebraicDP(
    F=NamedProduct({"capacity": Reals(unit="J")}),
    R=NamedProduct({"mass": Reals(unit="kg")}),
    equations={"mass": lambda f: f["capacity"] / 1.8e6},
    name="battery",
)
actuator = AlgebraicDP(
    F=NamedProduct({"lift_force": Reals(unit="N")}),
    R=NamedProduct({"power": Reals(unit="W")}),
    equations={"power": lambda f: 10.0 * f["lift_force"] ** 2},
    name="actuator",
)
battery, actuator

(DP(battery: capacity:R+[J] -> A[mass:R+[kg]]),
 DP(actuator: lift_force:R+[N] -> A[power:R+[W]]))

## Assembly

Declare the outer interface, add the subsystems, then attach connection constraints. Each constraint reads `target_port >= demand(ctx)` where `ctx` is a dict carrying the outer functionalities (`endurance`, `extra_payload`, `extra_power`) under their bare names and every subsystem R port under its dotted name (`battery.mass`, `actuator.power`).


In [3]:
G = 9.81

sys = System("drone")
sys.provides("endurance", unit="s")
sys.provides("extra_payload", unit="kg")
sys.provides("extra_power", unit="W")
sys.requires("total_mass", unit="kg")

sys.add("battery", battery)
sys.add("actuator", actuator)

sys.constrain(
    "battery.capacity",
    lambda x: (x["actuator.power"] + x["extra_power"]) * x["endurance"],
)
sys.constrain(
    "actuator.lift_force",
    lambda x: G * (x["battery.mass"] + x["extra_payload"]),
)
sys.constrain(
    "total_mass",
    lambda x: x["battery.mass"] + x["extra_payload"],
)
print(sys)

System('drone'):
  provides:
    endurance: R+[s]
    extra_payload: R+[kg]
    extra_power: R+[W]
  requires:
    total_mass: R+[kg]
  subsystems:
    battery: (capacity) -> (mass)
    actuator: (lift_force) -> (power)
  constraints:
    battery.capacity >= <demand>
    actuator.lift_force >= <demand>
    total_mass >= <demand>


## Build and solve

In [4]:
drone = sys.build()
print(drone)
print()

cases = [
    ("Short, light",   dict(endurance=60.0,   extra_payload=0.10, extra_power=1.0)),
    ("Medium, modest", dict(endurance=300.0,  extra_payload=0.50, extra_power=5.0)),
    ("Longer mission", dict(endurance=600.0,  extra_payload=0.50, extra_power=5.0)),
    ("Marginal",       dict(endurance=600.0,  extra_payload=1.00, extra_power=10.0)),
    ("Infeasible",     dict(endurance=1800.0, extra_payload=1.00, extra_power=10.0)),
]
for label, f in cases:
    result = solve(drone, f, max_iter=200)
    print(f"{label:<16} iters={result.iterations:>3}  "
          f"feasible={result.feasible}  {result.antichain}")

DP(loop(drone_inner, axis=__modules__): endurance:R+[s]×extra_payload:R+[kg]×extra_power:R+[W] -> A[total_mass:R+[kg]])

Short, light     iters= 17  feasible=True  Antichain[(total_mass=0.1004 kg)]
Medium, modest   iters= 43  feasible=True  Antichain[(total_mass=0.5492 kg)]
Longer mission   iters= 81  feasible=True  Antichain[(total_mass=0.6283 kg)]
Marginal         iters= 38  feasible=False  Antichain[(total_mass=⊤)]
Infeasible       iters= 22  feasible=False  Antichain[(total_mass=⊤)]


## What changed compared to notebook 01

The values are identical (e.g. for Medium, modest: `total_mass = 0.5492 kg = 0.04921 (battery) + 0.5 (payload)`). The iteration count is roughly 2x larger, because the loop now updates `battery.mass` and `actuator.power` in alternation rather than in one coupled step. Same fixed point, finer-grained updates.

The payoff is *modularity*: the battery and actuator are reusable building blocks. The next notebook (**08**) adds a third subsystem (a motor catalog with discrete choices) and produces a multi-point Pareto front from the same machinery.
